In [ ]:
# CELL 1: THƯ VIỆN & CẤU HÌNH (TỐI ƯU CHO GPU L4)
import os, glob, warnings, time, shutil
import numpy as np
import pandas as pd
from PIL import Image
import cv2
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score
import torch.optim as optim

warnings.filterwarnings('ignore')

class CFG:
    # --- ĐƯỜNG DẪN ---
    drive_folder = '/content/drive/MyDrive/APTOS 2019' # Thư mục chứa file ZIP trên Drive
    extract_dir  = '/content/dataset_full'             # Thư mục giải nén trên Colab
    csv_path     = '/content/full_labels.csv'
    save_dir     = '/content/drive/MyDrive/'           # Nơi lưu 5 file .pth và file OOF

    # --- THÔNG SỐ TRAINING ---
    img_size     = 512   # Đã nâng lên 512
    batch_size   = 16    # Tăng lên 16 nhờ VRAM 24GB của GPU L4
    epochs       = 20
    lr           = 3e-4
    weight_decay = 1e-4
    n_folds      = 5
    trn_folds    = [0, 1, 2, 3, 4]
    seed         = 42
    num_workers  = 4     # Tăng số luồng load data

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
seed_everything(CFG.seed)
print(f"Cấu hình sẵn sàng! Đang sử dụng thiết bị: {CFG.device}")

Cấu hình sẵn sàng! Đang sử dụng thiết bị: cuda


In [ ]:
# CELL 2: KHÔI PHỤC DỮ LIỆU TỪ GOOGLE DRIVE
def setup_dataset():
    # Kiểm tra xem dữ liệu đã được giải nén chưa (tránh giải nén lại khi restart cell)
    if os.path.exists(CFG.extract_dir) and len(os.listdir(CFG.extract_dir)) > 0:
        print(f"Dữ liệu đã sẵn sàng tại {CFG.extract_dir}. Bỏ qua bước giải nén.")
        return

    print(f" Đang quét thư mục Drive: {CFG.drive_folder}...")
    os.makedirs(CFG.extract_dir, exist_ok=True)

    # 1. Khôi phục file CSV
    csv_files = glob.glob(os.path.join(CFG.drive_folder, '*.csv'))
    if csv_files:
        shutil.copy(csv_files[0], CFG.csv_path)
        print(f"Đã copy file nhãn: {os.path.basename(csv_files[0])}")
    else:
        raise FileNotFoundError("Không tìm thấy file .csv nào trong Drive!")

    # 2. Khôi phục và giải nén các file ZIP
    sub_zips = glob.glob(os.path.join(CFG.drive_folder, '*.zip'))
    print(f"Tìm thấy {len(sub_zips)} file nén")

    local_temp_zip = '/content/temp_data.zip'
    for i, zip_path in enumerate(sub_zips, 1):
        print(f"Đang giải nén khối [{i}/{len(sub_zips)}]: {os.path.basename(zip_path)}")
        # Copy file zip từ Drive sang môi trường Colab (để tăng tốc độ đọc)
        shutil.copyfile(zip_path, local_temp_zip)

        # Dùng lệnh Linux unzip (nhanh hơn thư viện python zipfile rất nhiều)
        os.system(f"unzip -q -n '{local_temp_zip}' -d '{CFG.extract_dir}'")

        # Xóa file zip tạm để dọn dẹp ổ cứng Colab
        os.remove(local_temp_zip)

    print("Hoàn tất quá trình chuẩn bị ảnh!")

setup_dataset()

 Đang quét thư mục Drive: /content/drive/MyDrive/eyepacs_6k...
Đã copy file nhãn: trainLabels.csv
Tìm thấy 6 file nén
Đang giải nén khối [1/6]: train_6k.zip
Đang giải nén khối [2/6]: group_2_6k.zip
Đang giải nén khối [3/6]: group_3_6k.zip
Đang giải nén khối [4/6]: group_1_6k.zip
Đang giải nén khối [5/6]: group_4_6k.zip
Đang giải nén khối [6/6]: group_5_6k.zip
Hoàn tất quá trình chuẩn bị 35.000+ bức ảnh!


In [ ]:
# ==============================================================================
# CELL 2: TIỀN XỬ LÝ ẢNH & AUGMENTATION
# ==============================================================================
def apply_ben_graham(path, sigma=10, size=CFG.img_size):
    img = cv2.imread(path)
    if img is None: return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (size, size))
    img = cv2.addWeighted(img, 4, cv2.GaussianBlur(img, (0, 0), sigma), -4, 128)
    mask = np.zeros(img.shape, dtype=np.uint8)
    cv2.circle(mask, (size // 2, size // 2), int(size * 0.47), (1, 1, 1), -1)
    return Image.fromarray((img * mask).astype(np.uint8))

train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.1)) # Kỹ thuật che mắt mô hình
])

val_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
print("Transform sẵn sàng!")

Transform sẵn sàng!


In [ ]:
# CELL 3: DATASET & STRATIFIED K-FOLD
class DRDataset(Dataset):
    def __init__(self, df, image_paths, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        img_name = str(self.df.loc[idx, 'image']).strip().split('.')[0]
        label = self.df.loc[idx, 'level']
        if img_name not in self.image_paths: return None
        img = apply_ben_graham(self.image_paths[img_name])
        if img is None: return None
        if self.transform: img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.float32)

def safe_collate(batch):
    batch = [b for b in batch if b is not None]
    return torch.utils.data.dataloader.default_collate(batch) if batch else None

# Lấy đường dẫn ảnh
img_paths = {os.path.splitext(f)[0]: os.path.join(root, f)
             for root, _, files in os.walk(CFG.extract_dir)
             for f in files if f.lower().endswith(('.jpeg', '.jpg', '.png'))}

# Đọc CSV và chia Fold
df = pd.read_csv(CFG.csv_path)
df.columns = ['image', 'level']

Fold = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
df['fold'] = -1
for fold_idx, (train_idx, val_idx) in enumerate(Fold.split(df, df['level'])):
    df.loc[val_idx, 'fold'] = fold_idx

print(f"Đã chia xong {CFG.n_folds} Fold")

Đã chia xong 5 Fold


In [ ]:
# ==============================================================================
# CELL 4: XÂY DỰNG MÔ HÌNH VÀ CÁC HÀM TRAIN/EVAL (ĐÃ FIX LỖI NAN)
# ==============================================================================
def build_model():
    model = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.DEFAULT)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(512, 1)
    )
    return model

def train_one_epoch(model, dataloader, optimizer, criterion, scaler):
    model.train()
    total_loss = 0
    for batch in tqdm(dataloader, desc="Training", leave=False):
        if batch is None: continue # Bỏ qua nếu mẻ dữ liệu bị lỗi
        images, labels = batch[0].to(CFG.device), batch[1].to(CFG.device)

        optimizer.zero_grad()
        # Vẫn dùng Autocast lúc train vì đã có scaler bảo vệ
        with torch.amp.autocast('cuda'):
            outputs = model(images).view(-1)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate(model, dataloader, criterion):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            if batch is None: continue # Bỏ qua nếu mẻ dữ liệu bị lỗi
            images, labels = batch[0].to(CFG.device), batch[1].to(CFG.device)

            outputs = model(images).view(-1)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            # Kiểm tra nan, nếu có thì ép về 0 (để tránh crash toàn bộ)
            preds = outputs.cpu().float().numpy()
            preds = np.nan_to_num(preds, nan=0.0)

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    preds_rounded = np.clip(np.round(all_preds), 0, 4).astype(int)
    qwk = cohen_kappa_score(all_labels, preds_rounded, weights='quadratic')
    return total_loss / len(dataloader), qwk, all_preds

print("Engines đã được Fix lỗi NaN và sẵn sàng!")

Engines đã được Fix lỗi NaN và sẵn sàng!


In [ ]:
# ==============================================================================
# CELL 5: CHẠY 5-FOLD (ĐÃ THÊM WEIGHTED RANDOM SAMPLER ÉP HỌC ĐỒNG ĐỀU)
# ==============================================================================
import os
from torch.utils.data import WeightedRandomSampler # <--- Thư viện bốc bài tỷ lệ

oof_predictions = np.zeros(len(df))

for fold in CFG.trn_folds:
    print(f"\nFOLD {fold}\n{'='*40}")

    # 1. Chuẩn bị Dữ liệu
    train_df = df[df['fold'] != fold]
    val_df   = df[df['fold'] == fold]
    val_indices = val_df.index.values

    # --- TÍNH TOÁN WEIGHTED RANDOM SAMPLER Ở ĐÂY ---
    # 1a. Đếm số lượng ảnh của từng class (từ 0 đến 4) trong tập Train
    class_counts = train_df['level'].value_counts().sort_index().values

    # 1b. Tính trọng số cho từng class (Class nào ít ảnh thì trọng số càng TO)
    class_weights = 1.0 / class_counts

    # 1c. Gắn trọng số tương ứng cho từng bức ảnh cụ thể trong df
    sample_weights = [class_weights[label] for label in train_df['level'].values]

    # 1d. Khởi tạo Sampler
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(train_df), replacement=True)
    # -----------------------------------------------

    # 1e. Nạp Sampler vào DataLoader (LƯU Ý: Đã xóa shuffle=True)
    train_loader = DataLoader(DRDataset(train_df, img_paths, train_transforms), batch_size=CFG.batch_size, sampler=sampler, num_workers=CFG.num_workers, collate_fn=safe_collate, pin_memory=True)
    val_loader   = DataLoader(DRDataset(val_df, img_paths, val_transforms), batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, collate_fn=safe_collate, pin_memory=True)

    # 2. Khởi tạo Mô hình & Công cụ
    model = build_model().to(CFG.device)
    criterion = nn.SmoothL1Loss()
    optimizer = optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs)
    scaler    = torch.amp.GradScaler('cuda')

    # Đường dẫn lưu file
    best_path = os.path.join(CFG.save_dir, f'efficientnet_b4_fold{fold}_best.pth')
    last_path = os.path.join(CFG.save_dir, f'efficientnet_b4_fold{fold}_last.pth')

    best_qwk = -1.0
    best_val_preds = []
    start_epoch = 0

    # ---------------------------------------------------------
    # 3. KIỂM TRA VÀ KHÔI PHỤC TIẾN ĐỘ (RESUME CHECKPOINT)
    # ---------------------------------------------------------
    if os.path.exists(last_path):
        print(f"Progress Fold {fold}...")
        checkpoint = torch.load(last_path, map_location=CFG.device, weights_only=False)

        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        scaler.load_state_dict(checkpoint['scaler_state'])

        start_epoch = checkpoint['epoch'] + 1
        best_qwk = checkpoint['best_qwk']
        if 'best_val_preds' in checkpoint:
            best_val_preds = checkpoint['best_val_preds']

        print(f"Continue Epoch {start_epoch+1}. QWK record: {best_qwk:.4f}")
    else:
        print("Bắt đầu train từ Epoch 1.")

    # ---------------------------------------------------------
    # 4. VÒNG LẶP HUẤN LUYỆN
    # ---------------------------------------------------------
    for epoch in range(start_epoch, CFG.epochs):
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        vl_loss, vl_qwk, vl_preds = evaluate(model, val_loader, criterion)
        scheduler.step()

        print(f"Epoch {epoch+1}/{CFG.epochs} | Tr_Loss: {tr_loss:.4f} | Val_Loss: {vl_loss:.4f} | Val_QWK: {vl_qwk:.4f}")

        # A. LƯU BẢN CHÍNH THỨC (Khi phá kỷ lục)
        if vl_qwk > best_qwk:
            best_qwk = vl_qwk
            best_val_preds = vl_preds

            torch.save({'model_state': model.state_dict(), 'best_qwk': best_qwk}, best_path)
            print(f"Best record {os.path.basename(best_path)}")

        # B. LƯU BẢN NHÁP (Lưu sau MỌI epoch để chống móm)
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'scaler_state': scaler.state_dict(),
            'best_qwk': best_qwk,
            'best_val_preds': best_val_preds
        }, last_path)
        print(f"Saved record (Checkpoint).")

    # 5. Điền kết quả tốt nhất của Fold này vào bảng OOF
    oof_predictions[val_indices] = best_val_preds
    print(f"Hoàn tất Fold {fold}!")


# LƯU SỔ XUẤT RA FILE CHO NOTEBOOK 2
df['oof_preds'] = oof_predictions
oof_path = os.path.join(CFG.save_dir, 'oof_preds_b4.csv')
df.to_csv(oof_path, index=False)

print(f"File ghi chép điểm thô OOF đã xuất ra: {oof_path}")


FOLD 0
Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [00:00<00:00, 238MB/s]


Progress Fold 0...
Continue Epoch 21. QWK record: 0.8262
Hoàn tất Fold 0!

FOLD 1
Progress Fold 1...
Continue Epoch 21. QWK record: 0.8324
Hoàn tất Fold 1!

FOLD 2
Progress Fold 2...
Continue Epoch 21. QWK record: 0.8237
Hoàn tất Fold 2!

FOLD 3
Progress Fold 3...
Continue Epoch 21. QWK record: 0.8241
Hoàn tất Fold 3!

FOLD 4
Progress Fold 4...
Continue Epoch 19. QWK record: 0.8202


Epoch 19/20 | Tr_Loss: 0.0631 | Val_Loss: 0.1184 | Val_QWK: 0.8182
Saved record (Checkpoint).


Epoch 20/20 | Tr_Loss: 0.0617 | Val_Loss: 0.1187 | Val_QWK: 0.8210
Best record efficientnet_b4_fold4_best.pth
Saved record (Checkpoint).
Hoàn tất Fold 4!
File ghi chép điểm thô OOF đã xuất ra: /content/drive/MyDrive/oof_preds_b4.csv
